In [ ]:
# parameters and config
# Edit these values directly in Fabric, or pass them from a Fabric Data Pipeline.
ENV = "dev"
SCHEMA = "dbo"
MARKET = "AU"
MOCK_IPSTACK = True
LOOKBACK_DAYS = 90
LISTINGS_FILE = "listings.csv"
VIEWS_FILE = "property_views_5k.csv"

try:
    from notebookutils import mssparkutils
except ImportError:
    mssparkutils = None

from pyspark.sql import SparkSession
spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()

class Config:
    def __init__(self):
        self.env = ENV
        self.schema = SCHEMA
        self.market = MARKET
        self.mock_ipstack = MOCK_IPSTACK
        self.lookback_days = LOOKBACK_DAYS
        self.listings_file = LISTINGS_FILE
        self.views_file = VIEWS_FILE
        self.files_base = "Files"  # Fabric attached Lakehouse path
        self.listings_path = f"{self.files_base}/raw/sample/{self.listings_file}"
        self.views_path = f"{self.files_base}/raw/sample/{self.views_file}"
        self.fixture_base = f"{self.files_base}/fixtures/ipstack"

        self.bronze_listings = "bronze_listings"
        self.bronze_property_views = "bronze_property_views"
        self.bronze_ipstack_raw = "bronze_ipstack_raw"
        self.bronze_ipstack_errors = "bronze_ipstack_errors"
        self.silver_listings = "silver_listings"
        self.silver_ip_dim = "silver_ip_dim"
        self.silver_visits_enriched = "silver_visits_enriched"
        self.gold_suburb_interest = "gold_suburb_interest"
        self.gold_property_type_by_suburb = "gold_property_type_by_suburb"
        self.gold_price_engagement = "gold_price_engagement"
        self.gold_conversion_gaps = "gold_conversion_gaps"
        self.gold_property_trends = "gold_property_trends"
        self.gold_region_type_preference = "gold_region_type_preference"
        self.gold_repeat_interest = "gold_repeat_interest"
        self.gold_interstate_flow = "gold_interstate_flow"

    def table_fqn(self, table: str) -> str:
        return f"{self.schema}.{table}"

conf = Config()
print(f"env={conf.env} market={conf.market} mock={conf.mock_ipstack} views={conf.views_file}")
print(f"listings_path={conf.listings_path}")
print(f"views_path={conf.views_path}")




In [ ]:
import time

class SetupHelper:
    def __init__(self):
        self.c = conf

    def create_tables(self):
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {self.c.schema}")
        T = self.c.table_fqn
        spark.sql(f"""CREATE TABLE IF NOT EXISTS {T(self.c.bronze_listings)} (
          property_id STRING, property_type STRING, bedrooms INT, bathrooms INT,
          suburb STRING, postcode STRING, state STRING, price INT, price_range STRING,
          is_luxury BOOLEAN, land_size_sqm INT, _ingested_at TIMESTAMP
        ) USING DELTA""")
        spark.sql(f"""CREATE TABLE IF NOT EXISTS {T(self.c.bronze_property_views)} (
          event_id STRING, session_id STRING, event_ts TIMESTAMP, event_type STRING,
          user_id STRING, ip STRING, user_agent STRING, property_id STRING,
          view_duration_sec INT, enquiry_flag BOOLEAN, favorite_flag BOOLEAN,
          event_date DATE, _ingested_at TIMESTAMP
        ) USING DELTA PARTITIONED BY (event_date)""")
        spark.sql(f"""CREATE TABLE IF NOT EXISTS {T(self.c.bronze_ipstack_raw)} (
          ip STRING, response_json STRING, http_status INT, api_called_at TIMESTAMP
        ) USING DELTA""")
        spark.sql(f"""CREATE TABLE IF NOT EXISTS {T(self.c.bronze_ipstack_errors)} (
          ip STRING, error_message STRING, http_status INT, api_called_at TIMESTAMP
        ) USING DELTA""")
        spark.sql(f"""CREATE TABLE IF NOT EXISTS {T(self.c.silver_listings)} (
          property_id STRING, property_type STRING, bedrooms INT, bathrooms INT,
          suburb STRING, postcode STRING, state STRING, price INT, price_range STRING,
          is_luxury BOOLEAN, land_size_sqm INT
        ) USING DELTA""")
        spark.sql(f"""CREATE TABLE IF NOT EXISTS {T(self.c.silver_ip_dim)} (
          ip STRING, country_code STRING, country_name STRING, visitor_region STRING,
          visitor_city STRING, latitude DOUBLE, longitude DOUBLE, timezone_id STRING,
          isp STRING, is_vpn BOOLEAN, enriched_at TIMESTAMP
        ) USING DELTA""")
        spark.sql(f"""CREATE TABLE IF NOT EXISTS {T(self.c.silver_visits_enriched)} (
          event_id STRING, session_id STRING, event_ts TIMESTAMP, user_id STRING, ip STRING,
          property_id STRING, view_duration_sec INT, enquiry_flag BOOLEAN, favorite_flag BOOLEAN,
          event_date DATE, visitor_country STRING, visitor_region STRING, visitor_city STRING,
          visitor_state STRING, timezone_id STRING, device_type STRING,
          suburb STRING, listing_state STRING, property_type STRING, price_range STRING,
          is_luxury BOOLEAN, price INT
        ) USING DELTA PARTITIONED BY (event_date)""")

    def setup(self):
        t0 = time.time()
        self.create_tables()
        print(f"✅ Setup completed in {int(time.time()-t0)}s")

SetupHelper().setup()




